# Calling `fpm_rs.plot` directly
Build each diagnostic figure explicitly, inspect the named axes that come back, and customize Matplotlib titles before displaying the figures inline. Before running the code, install the released package in this notebook's environment with `%pip install "fpm-rs[notebook]"`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

import fpm_rs as fpm

In [ ]:
optics = fpm.Optics(532e-9, 0.10, 4.0, 6.5e-6)
geometry = fpm.PlanarLEDArray((3, 3), 4e-3, (1.0, 1.0), fpm.ArrayPose.from_translation((0.0, 0.0, -90e-3)))
illumination = fpm.Illumination(geometry)
model = fpm.compile_model(optics, illumination, (16, 16), (32, 32))

row, column = np.indices(model.reconstruction_shape)
amplitude = 0.7 + 0.3 * ((row // 4 + column // 4) % 2)
phase = 0.3 * np.sin(row / 5.0) * np.cos(column / 7.0)
object_field = np.asarray(amplitude * np.exp(1j * phase), dtype=np.complex128)

simulation = fpm.simulate(model, object_field, seed=31)
problem = fpm.ReconstructionProblem(
    simulation.measurements,
    simulation.reconstruction_model,
    name="diagnostics-plots",
)
recorder = fpm.DiagnosticRecorder("debug", every=2)
result = fpm.AlternatingProjection(iterations=6).run(
    problem,
    callbacks=[recorder],
)
diagnostics = recorder.diagnostics()
result.runtime.completed_iterations, result.final_objective

In [ ]:
plot_results = {
    "convergence": fpm.plot.plot_convergence(diagnostics),
    "frame_residuals": fpm.plot.plot_frame_residuals(diagnostics),
    "raw_stack_stats": fpm.plot.plot_raw_stack_stats(diagnostics),
    "fourier_coverage": fpm.plot.plot_fourier_coverage(diagnostics),
    "crop_indices": fpm.plot.plot_crop_indices(diagnostics),
    "residuals_on_fourier_centers": (
        fpm.plot.plot_residuals_on_fourier_centers(diagnostics)
    ),
}
plot_results["convergence"][1]["objective"].set_title("Custom convergence view")
plot_results["frame_residuals"][1]["residual_magnitude"].set_title("Residual overview")
plot_results["crop_indices"][1]["crop_indices"].set_title(
    "Fourier crops used by the model"
)
plot_results["convergence"][1].keys()

In [ ]:
for plot_name, plot in plot_results.items():
    if plot is None:
        continue
    figure, axes = plot
    display(figure)
    # figure.savefig(f"{plot_name}.png", dpi=180, bbox_inches="tight")
    plt.close(figure)

In [ ]:
diagnostics["iteration_diagnostics"][-1]